In [12]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[]


In [9]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
from datetime import datetime

In [10]:


# Mixup and Cutmix augmentations

def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    H, W = x.size(2), x.size(3)
    cx, cy = np.random.randint(W), np.random.randint(H)
    w, h = int(W * np.sqrt(1 - lam)), int(H * np.sqrt(1 - lam))
    x1, y1 = max(cx - w // 2, 0), max(cy - h // 2, 0)
    x2, y2 = min(cx + w // 2, W), min(cy + h // 2, H)
    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def train_mobilenet_v3(data_dir, num_epochs=200, batch_size=32, learning_rate=3e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load MobileNetV3-Large
    model = torch.hub.load('pytorch/vision:v0.10.0', 'mobilenet_v3_large', pretrained=True)
    for param in model.parameters():
        param.requires_grad = True
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, 2)
    model.to(device)

    # Data loading
    dataset = datasets.ImageFolder(root=data_dir)
    labels = [s[1] for s in dataset.samples]
    train_idx, test_idx = train_test_split(np.arange(len(dataset)), test_size=0.2, stratify=labels, random_state=42)
    train_idx, val_idx = train_test_split(train_idx, test_size=0.25, stratify=[labels[i] for i in train_idx], random_state=42)

    # Class weights for balancing
    class_counts = Counter([labels[i] for i in train_idx])
    class_weights = torch.tensor([class_counts[0], class_counts[1]], dtype=torch.float32).to(device)
    class_weights = 1.0 / class_weights / class_weights.sum()

    # Data augmentation
    train_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(30),
        transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
        transforms.GaussianBlur(5, sigma=(0.1, 0.5)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    test_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Split datasets
    dataset.transform = train_transform
    train_dataset = Subset(dataset, train_idx)
    dataset.transform = test_transform
    val_dataset, test_dataset = Subset(dataset, val_idx), Subset(dataset, test_idx)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    # Loss, optimizer, scheduler
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

    train_losses, val_losses = [], []
    train_accs, val_accs, val_f1s = [], [], []

    for epoch in range(num_epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        train_losses.append(total_loss / len(train_loader))
        train_accs.append(correct / total)
        print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_losses[-1]:.4f}, Train Acc: {train_accs[-1]:.4f}")
        scheduler.step(total_loss / len(train_loader))

    # Save model
    save_path = f"mobilenetv3_model.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to {save_path}")

    # Plotting metrics
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(train_accs, label='Train Accuracy')
    plt.legend()
    plt.show()

# Example usage
# train_mobilenet_v3('path_to_your_dataset')


In [11]:

if __name__ == "__main__":
    data_dir = "dataset" #Ensure this directory contains 'healthy' and 'infected' folders
    torch.cuda.empty_cache()
    model = train_mobilenet_v3(data_dir)

Using cache found in C:\Users\augus/.cache\torch\hub\pytorch_vision_v0.10.0
c:\Augustine\Major Project\healthyCow 0.0\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Augustine\Major Project\healthyCow 0.0\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Augustine\Major Project\healthyCow 0.0\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/200, Train Loss: 0.2618, Train Acc: 0.9712
Epoch 2/200, Train Loss: 0.2102, Train Acc: 0.9971
Epoch 3/200, Train Loss: 0.2046, Train Acc: 0.9992
Epoch 4/200, Train Loss: 0.2032, Train Acc: 1.0000
Epoch 5/200, Train Loss: 0.2014, Train Acc: 1.0000
Epoch 6/200, Train Loss: 0.2017, Train Acc: 1.0000


KeyboardInterrupt: 